# Mobility Robustness Optimization

**to be rewritten explaining the purpose of MRO in short**

Takes in new observation data to train or update the bayesian digital twin model. It processes the input data and updates the model to better reflect the current network conditions.

Then MRO optimizes the mobility robustness by solving the underlying problem using the trained model: finding optimal HYST and TTT. It iteratively improves the network performance over a specified number of epochs. There are two solve approach shown: simple MRO and Reinforced MRO.

In [ ]:
import sys
from pathlib import Path
sys.path.append(f"{Path().absolute().parent}")

In [ ]:
import pandas as pd
from apps.mobility_robustness_optimization.simple_mro import SimpleMRO
from apps.mobility_robustness_optimization.mro_rl import ReinforcedMRO
from notebooks.radp_library import get_ue_data

## Example Data

- `new_data`: used as input data to be fed into MRO. Use your own data if needed.
- `topology`: to be fed into MRO as well. Use your own data if needed.
- `mobility_model` itself? or `mobility_model_params` to be used? #FIXME

In [ ]:
new_data = pd.read_csv('data/sim_data/UE_data_20UE_100ticks.csv')
topology = pd.read_csv('data/sim_data/topology.csv')

# TODO: need to use smaller data and add these data files to the repo in something like data/mro_data, sim_data/ is gitignored.

In [ ]:
topology.loc[topology["cell_id"] == "cell_1", "cell_lat"] = -90
topology.loc[topology["cell_id"] == "cell_2", "cell_lat"] = 0
topology.loc[topology["cell_id"] == "cell_3", "cell_lat"] = 90

topology.loc[topology["cell_id"] == "cell_1", "cell_lon"] = -180
topology.loc[topology["cell_id"] == "cell_2", "cell_lon"] = 0
topology.loc[topology["cell_id"] == "cell_3", "cell_lon"] = 180

topology.loc[topology["cell_id"] == "cell_1", "cell_carrier_freq_mhz"] = 2100
topology.loc[topology["cell_id"] == "cell_2", "cell_carrier_freq_mhz"] = 2100
topology.loc[topology["cell_id"] == "cell_3", "cell_carrier_freq_mhz"] = 2100

new_data.drop(columns=['mock_ue_id', 'tick'], inplace=True)

# TODO: dump the final version of these data and get rid of this preprocessing step

In [ ]:
params = {
    "ue_tracks_generation": {
            "params": {
                "simulation_duration": 3600,
                "simulation_time_interval_seconds": 0.01,
                "num_ticks": 100,
                "num_batches": 1,
                "ue_class_distribution": {
                    "stationary": {
                        "count": 5,
                        "velocity": 0,
                        "velocity_variance": 1
                    },
                    "pedestrian": {
                        "count": 5,
                        "velocity": 2,
                        "velocity_variance": 1
                    },
                    "cyclist": {
                        "count": 5,
                        "velocity": 5,
                        "velocity_variance": 1
                    },
                    "car": {
                        "count": 5,
                        "velocity": 20,
                        "velocity_variance": 1
                    }
                },
                "lat_lon_boundaries": {
                    "min_lat": -90,
                    "max_lat": 90,
                    "min_lon": -180,
                    "max_lon": 180
                },
                "gauss_markov_params": {
                    "alpha": 0.5,
                    "variance": 0.8,
                    "rng_seed": 42,
                    "lon_x_dims": 100,
                    "lon_y_dims": 100,
                    "// TODO": "Account for supporting the user choosing the anchor_loc and cov_around_anchor.",
                    "// Current implementation": "the UE Tracks generator will not be using these values.",
                    "// anchor_loc": {},
                    "// cov_around_anchor": {}
            }
        }
    }
}

# ? As paul said, may be need to create mobility model for this param and then feed into MRO directly. Or should we keep it as is?
# TODO: Decide upon the above question and implement the mobility model if needed.

## Simple MRO

### No rx power data

- has `new_data` with no rx power data, only [latitude, longitude]
- in this case, call `preprocess_ue_data(new_data, topology)` from `radp_library.py` to get Free-space Path Loss (FSPL) calculated rx power data, in cartesian format.

In [ ]:
from notebooks.radp_library import preprocess_ue_data

input_data = preprocess_ue_data(new_data, topology)

#### Initialize MRO

In [ ]:
mro = SimpleMRO(mobility_model_params = params, topology = topology)

#### Update Call 1: trains bdt from scratch

In [ ]:
mro.bayesian_digital_twins

In [ ]:
mro.train_or_update_rf_twin(input_data)

In [ ]:
mro.bayesian_digital_twins

In [ ]:
mro.save_bdt()

#### Call Solve

In [ ]:
# adjust n_epochs for better performance
mro.solve(n_epochs=3)

In [ ]:
# loading new UE data assuming user have new observations now.
new_ue = get_ue_data(params)
new_ue.rename(columns={"lon": "longitude", "lat": "latitude"}, inplace=True)

# calling the preprocess_ue_data function to get rx power data in cartesian
new_data2 = preprocess_ue_data(new_ue, topology)

#### Update Call 2: updates bdt

In [ ]:
mro.train_or_update_rf_twin(new_data2)

# ! ISSUE: between 1st and 2nd update call: either _prediction() or solve() call required or throws: "An unexpected error occurred: Fantasy observations can only be added after making predictions with a model so that all test independent caches exist. Call the model on some data first!"

In [ ]:
mro.bayesian_digital_twins

In [ ]:
mro.save_bdt()

In [ ]:
# adjust n_epochs for better performance
mro.solve(n_epochs=3)

#### load bdt and update

In [ ]:
mro_new = SimpleMRO(mobility_model_params = params, topology = topology)

In [ ]:
mro_new.bayesian_digital_twins

In [ ]:
mro_new.load_bdt()

In [ ]:
mro_new.bayesian_digital_twins

In [ ]:
mro_new.train_or_update_rf_twin(new_data2)

## RL MRO

In [ ]:
rl_mro = ReinforcedMRO(mobility_model_params = params, topology = topology)

In [ ]:
input_data = preprocess_ue_data(new_data, topology)

In [ ]:
rl_mro.train_or_update_rf_twin(input_data)

In [ ]:
rl_mro.solve()